# PII 마스킹

명시적으로 만든 합성 개인정보를 모델 호출과 감사 로그 전에 가립니다.

In [1]:
%pip install -q langchain==1.4.0 langchain-openai==1.6.1 python-dotenv==1.2.3

Note: you may need to restart the kernel to use updated packages.


## 실행 환경 준비

실제 사람의 정보는 사용하지 않습니다. 예상 API 비용은 0.01달러 미만입니다.

In [2]:
from pathlib import Path
import os, sys
from dotenv import find_dotenv, load_dotenv

start = Path.cwd().resolve()
candidates = (start, *start.parents)
ROOT = next(path for path in candidates if (path / "labs/part4/ch16-guardrails").is_dir())
LAB = ROOT / "labs/part4/ch16-guardrails"
load_dotenv(find_dotenv(usecwd=True))
assert os.getenv("OPENAI_API_KEY"), "프로젝트 루트 .env에 OPENAI_API_KEY가 필요합니다."
print("Python:", sys.version.split()[0], "| API 키 준비: True")

Python: 3.12.13 | API 키 준비: True


In [3]:
from langchain.chat_models import init_chat_model

model = init_chat_model(
    "openai:gpt-5.4-mini", temperature=0, max_tokens=140,
    timeout=30, max_retries=0,
)
print("모델 준비:", type(model).__name__)

모델 준비: ChatOpenAI


## 기본 실행

PII(personally identifiable information)는 개인을 알아볼 수 있게 하는 정보입니다.

In [4]:
import re

def mask_pii(text: str) -> tuple[str, dict[str, int]]:
    patterns = {
        "email": r"(?<![\w.])[\w.+-]+@[\w.-]+\.[A-Za-z]{2,}(?![\w.])",
        "phone": r"(?<!\d)01[016789][ -]?\d{3,4}[ -]?\d{4}(?!\d)",
    }
    masked, counts = text, {}
    for name, pattern in patterns.items():
        masked, counts[name] = re.subn(pattern, f"<{name.upper()}_MASKED>", masked)
    return masked, counts

synthetic = "문의자는 test.user@example.com, 010-1234-5678로 답변을 요청했습니다."
masked, counts = mask_pii(synthetic)
print("마스킹:", masked)
print("종류별 개수:", counts)

마스킹: 문의자는 <EMAIL_MASKED>, <PHONE_MASKED>로 답변을 요청했습니다.
종류별 개수: {'email': 1, 'phone': 1}


## 모델 입력 확인

모델에는 마스킹된 문장만 전달합니다.

In [5]:
response = model.invoke([
    {"role": "system", "content": "입력에 남은 연락 수단의 종류만 말하세요."},
    {"role": "user", "content": masked},
])
print("모델 답:", response.text)
print("원문 이메일 재노출:", "test.user@example.com" in response.text)

모델 답: 이메일, 전화번호
원문 이메일 재노출: False


## 실패 재현

원문을 로그에 쓰면 개인 식별 문자열이 그대로 남습니다.

In [6]:
unsafe_log = {"query": synthetic}
print("원문 이메일 포함:", "test.user@example.com" in unsafe_log["query"])
assert "test.user@example.com" in unsafe_log["query"]

원문 이메일 포함: True


## 수정

마스킹 결과와 탐지 개수만 기록합니다.

In [7]:
safe_log = {"query_masked": masked, "pii_counts": counts}
print(safe_log)
assert "test.user@example.com" not in __import__('json').dumps(safe_log)

{'query_masked': '문의자는 <EMAIL_MASKED>, <PHONE_MASKED>로 답변을 요청했습니다.', 'pii_counts': {'email': 1, 'phone': 1}}
